# Phase Lines: Reading the Future Off the Equation

**Time:** about 15 to 20 minutes.
**You need:** nothing but this page. New to notebooks? Do `00-start-here.ipynb` first.

Most of the equations we have modeled so far have the form

$$u'=f(u),$$

where the rate of change depends only on where you are right now, not on what time it is. A fish
population grows at a rate set by how many fish there are, not by the calendar. Equations like this
are called **autonomous**, and they have a property that makes them unusually easy to read: you can
predict every solution's long-run fate from the graph of $f$ alone.

> **The main idea:** solutions stand still wherever $f(u)=0$, and in between, the sign of $f$ says
> which way they move.

Everything in this activity is a consequence of that one sentence, including two different ways a
fishery can die.

Each section asks you to predict before you run anything. Commit to an answer first, on paper or
out loud. Being wrong and seeing why is the point.

## Run this first

The two cells below set up the drawing tools. You never need to read or change them. Run them
once each (**Shift + Enter**) and give them a moment on the first run.

In [ ]:
%pip install -q ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def trace(f, t0, y0, t_range=(-3, 3), steps=600, bounds=(-1e6, 1e6)):
    'Follow the slope field from (t0, y0) in both directions. Returns (ts, ys).'
    def march(t_start, t_end):
        n = max(int(steps * abs(t_end - t_start) / 6), 2)
        h = (t_end - t_start) / n
        ts, ys = [t_start], [y0]
        t, y = t_start, y0
        for _ in range(n):
            k1 = f(t, y)
            k2 = f(t + h/2, y + h*k1/2)
            k3 = f(t + h/2, y + h*k2/2)
            k4 = f(t + h, y + h*k3)
            y = y + h*(k1 + 2*k2 + 2*k3 + k4)/6
            t = t + h
            ts.append(t); ys.append(y)
            # Stop once the solution leaves the picture (blow up, or extinction).
            if not (bounds[0] <= y <= bounds[1]):
                break
        return ts, ys

    tb, yb = march(t0, t_range[0])
    tf, yf = march(t0, t_range[1])
    ts = np.array(tb[::-1] + tf[1:])
    ys = np.array(yb[::-1] + yf[1:])
    return ts, ys


def fixed_points(f, u_range):
    'Find where f(u) = 0 in u_range and classify each one. Returns [(u*, kind)].'
    u = np.linspace(*u_range, 20001)
    F = f(u)
    found = []
    # Places where f changes sign.
    for i in np.where(F[:-1] * F[1:] < 0)[0]:
        found.append(u[i] - F[i] * (u[i+1] - u[i]) / (F[i+1] - F[i]))
    # Places where f touches zero without crossing.
    A = np.abs(F)
    for i in range(1, len(u) - 1):
        if A[i] < 1e-6 and A[i] <= A[i-1] and A[i] <= A[i+1]:
            found.append(u[i])
    found.sort()
    points = []
    for p in found:
        if points and abs(p - points[-1][0]) < 0.01:
            continue
        e = 1e-3 * (u_range[1] - u_range[0])
        below, above = f(p - e), f(p + e)
        if below > 0 and above < 0:
            kind = 'stable'
        elif below < 0 and above > 0:
            kind = 'unstable'
        else:
            kind = 'semi-stable, from below' if below > 0 else 'semi-stable, from above'
        points.append((0.0 if abs(p) < 1e-6 else p, kind))
    return points


def portrait(f, u_range, starts, t_end, title, f_lim=2.0, physical_min=None, floor=None):
    'Left: f(u) on its side with the phase line. Right: solutions u(t). Shared u axis.'
    fig, (axf, axt) = plt.subplots(1, 2, figsize=(10, 5), sharey=True,
                                   gridspec_kw={'width_ratios': [1, 2]})
    u = np.linspace(*u_range, 800)
    points = fixed_points(f, u_range)

    # Left panel: the graph of f, turned so u runs up the page.
    axf.plot(f(u), u, color='0.2', linewidth=2)
    axf.axvline(0, color='0.5', linewidth=1)
    axf.set_xlim(-f_lim, f_lim)
    axf.set_xlabel("f(u) = u'   (left: falling, right: rising)")
    axf.set_ylabel('u')
    axf.set_title('f(u) on its side, with the phase line')
    axf.grid(alpha=0.25)

    # Phase line arrows between fixed points.
    edges = [u_range[0]] + [p for p, _ in points] + [u_range[1]]
    for lo, hi in zip(edges[:-1], edges[1:]):
        mid, half = (lo + hi) / 2, 0.3 * (hi - lo)
        if half < 0.05:
            continue
        s = f(mid)
        if s == 0:
            continue
        a, b = (mid - half, mid + half) if s > 0 else (mid + half, mid - half)
        axf.annotate('', xy=(0, b), xytext=(0, a),
                     arrowprops=dict(arrowstyle='-|>', color='C0', lw=2.5,
                                     mutation_scale=18))

    # Fixed points: filled = stable, open = unstable, half filled = semi-stable.
    for p, kind in points:
        style = dict(marker='o', markersize=12, markeredgecolor='black', color='black',
                     markeredgewidth=1.5, linestyle='none', zorder=5)
        if kind == 'stable':
            axf.plot(0, p, markerfacecolor='black', **style)
        elif kind == 'unstable':
            axf.plot(0, p, markerfacecolor='white', **style)
        else:
            axf.plot(0, p, fillstyle='bottom' if 'below' in kind else 'top',
                     markerfacecoloralt='white', **style)
        for ax in (axf, axt):
            ax.axhline(p, color='0.4', linestyle=':', linewidth=1.2)

    # Right panel: actual solutions from several starting values.
    lo_b = floor if floor is not None else u_range[0] - 1
    for i, u0 in enumerate(starts):
        ts, us = trace(lambda t, y: f(y), 0.0, float(u0), t_range=(0, t_end),
                       steps=max(600, int(100 * t_end)),
                       bounds=(lo_b, u_range[1] + 1))
        axt.plot(ts, us, linewidth=2.2, color=f'C{i % 10}',
                 linestyle=['-', '--', '-.', ':'][i % 4], label=f'u(0) = {u0:g}')
    axt.set_xlim(0, t_end)
    axt.set_xlabel('t')
    axt.set_title('solutions u(t)')
    axt.grid(alpha=0.25)
    axt.legend(loc='upper right', framealpha=0.9, fontsize=9)

    if physical_min is not None:
        for ax in (axf, axt):
            ax.axhspan(u_range[0], physical_min, color='0.88', zorder=0)
        axt.text(0.4, (u_range[0] + physical_min) / 2, 'not physical (negative fish)',
                 va='center', fontsize=9, color='0.3')

    axf.set_ylim(u_range)
    summary = ',   '.join(f'u = {p:.2f} ({k})' for p, k in points) or 'none in view'
    fig.suptitle(f'{title}\nfixed points: {summary}', fontsize=11)
    fig.tight_layout()
    plt.show()
    plt.close(fig)


print('Ready.')

## 1. The fishery, read off the equation

In class you met the Atlantic cod off Georges Bank, growing logistically and harvested at a fixed
fraction $0.2$ of the population each year. After simplifying, the model was

$$u'=0.05\,u\,(6-u),$$

with $u$ in thousands of fish and $t$ in years. Read it as a sentence: *the population grows at a
rate set by two things at once, how many fish there are and how much room is left below 6.*

Before running anything, predict:

1. At what values of $u$ does the population stop changing?
2. Between $0$ and $6$, is $u$ rising or falling? Above $6$?
3. A population that starts at $1$ thousand, and one that starts at $9$ thousand. Where does each
   end up?
4. Can a solution that starts below $6$ ever climb above $6$?

**How to read the picture.** The left panel is the graph of $f(u)=0.05u(6-u)$, turned on its side
so that $u$ runs up the page in both panels. Where the curve sits to the right of the center line,
$f>0$ and $u$ is rising; to the left, $u$ is falling. The arrows on the center line are the
**phase line**. A filled dot is a stable fixed point and an open dot is unstable. The right panel
shows real solutions $u(t)$, and the dotted lines run straight across from each fixed point.

In [ ]:
portrait(lambda u: 0.05*u*(6 - u), u_range=(-1, 13), starts=[0, 1, 3, 9, 12],
         t_end=25, f_lim=2.5, physical_min=0, title="u' = 0.05u(6 - u)")

### What to notice

Look straight across from any height. If the sideways graph is to the right of the center line at
that height, every curve on the right is climbing as it passes through that height. The phase line
is the right panel squashed flat: it throws away *when* things happen and keeps *where they go*.

<details>
<summary>Check yourself after you have written something</summary>

(1) $u=0$ and $u=6$, where $f(u)=0$. (2) Rising between $0$ and $6$, falling above $6$. (3) Both
end up at $6$, one climbing and one falling. That is why $6$ gets a filled dot: arrows point in
from both sides. (4) No. To get above $6$ a curve would have to pass through height $6$, and at
that height the equation says the rate of change is zero, so it can only approach the dotted line,
never cross it. The fixed points act as walls that divide the solutions into separate groups.
</details>

## 2. Turn up the harvest

The class question was whether taking $20\%$ a year is sustainable. The question a fishery manager
actually faces is *how far can we push it?* Keep the growth rate $r=0.5$ and the carrying capacity
$K=10$, and let the harvest fraction $h$ vary:

$$u'=0.5\,u\Big(1-\frac{u}{10}\Big)-h\,u.$$

Read it as a sentence: *the fish grow logistically, and every year you remove the fraction $h$ of
whatever is there.*

Before you touch the slider, predict:

1. As $h$ increases, what happens to the level the fishery settles at?
2. At what value of $h$ does that level reach zero?
3. Something subtler. Close to that value of $h$, do populations settle quickly or slowly?

Then run the cell and push $h$ to **both ends** of its range on purpose, then back toward the
middle. The gray band is negative fish, which cannot happen, but watch what the fixed points do
there anyway.

In [ ]:
from ipywidgets import interact, FloatSlider

def explore_fraction(h=0.2):
    h = round(h, 2)
    f = lambda u: 0.5*u*(1 - u/10) - h*u
    portrait(f, u_range=(-2, 12.5), starts=[1, 4, 9, 12], t_end=40, f_lim=2,
             physical_min=0, title=f"u' = 0.5u(1 - u/10) - {h:.2f}u   (harvest fraction h = {h:.2f})")

interact(
    explore_fraction,
    h=FloatSlider(value=0.2, min=0.0, max=0.8, step=0.05,
                  description='h', continuous_update=False)
);

### What changed, and what stayed the same?

**In your notes,** fill these in from the slider:

- At $h=0$ the fishery settles at ________. At $h=0.25$, at ________. At $h=0.45$, at ________.
- The fishery is gone for good once $h$ reaches ________.
- At $h=0.45$, are the curves done settling by year $40$?

<details>
<summary>Check yourself after you have written something</summary>

The settled level is $10-20h$: that is $10$, then $5$, then $1$. It slides down steadily and hits
zero at $h=0.5$, which is the growth rate $r$. Past that, the positive fixed point has slid into the
gray band where it means nothing, and $u=0$ has turned into a filled dot: every population heads to
extinction.

At $h=0.45$ the curves are still creeping at year $40$. Near the threshold the sideways graph
barely leans away from the center line, so the pull toward the fixed point is weak and recovery
from any disturbance takes a very long time. Ecologists call this <strong>critical slowing
down</strong>, and they look for it in real data as an early warning sign of collapse.

At exactly $h=0.50$ the two fixed points sit on top of each other at $0$ and the dot is half
filled. Remember that half-filled dot. It comes back.
</details>

What stayed the same is the shape of the story: until the threshold, one stable level and one
unstable one, with the stable level shrinking smoothly. This fishery fades. It does not crash. You
found the exact collapse threshold without solving anything.

## 3. When the shortcut goes silent

Now a small equation with a strange fixed point:

$$u'=(u-2)^2.$$

Read it as a sentence: *the rate of change is the squared distance from 2.* A square is never
negative, so this equation never lets $u$ decrease.

The fixed point is $u=2$. The shortcut from class says to check the sign of $f'$ there, and
$f'(u)=2(u-2)$ gives $f'(2)=0$. The shortcut is silent. So go back to the sign of $f$ itself.

Before running, predict:

1. A solution that starts at $u(0)=1.5$. Rising or falling? Where does it head?
2. A solution that starts at $u(0)=2.3$. Rising or falling? Where does it head?
3. Is $u=2$ stable? What would you draw on the phase line?

In [ ]:
portrait(lambda u: (u - 2)**2, u_range=(-1, 5), starts=[-0.5, 1, 1.5, 2, 2.3, 2.8],
         t_end=6, f_lim=4, title="u' = (u - 2)^2")

### What to notice

Look at the left panel. The sideways graph comes down, **touches** the center line at $u=2$, and
goes back the way it came without crossing. Both arrows point up. That touch without a crossing is
the picture of a semi-stable point, and it is why the dot is half filled: the filled half is the
side that gets pulled in.

<details>
<summary>Check yourself after you have written something</summary>

(1) Rising, toward $2$, more and more slowly, never quite arriving. (2) Rising, away from $2$, and
faster and faster: it leaves the top of the picture in a finite amount of time. (3) Solutions below
are pulled in and solutions above are pushed away, so $2$ is semi-stable. By the formal definition
that means <strong>unstable</strong>: a nudge upward of any size, however small, and the solution
never comes back.
</details>

## 4. A quota instead of a fraction

Real fisheries are usually not managed by taking a percentage. They are managed by a **quota**: a
fixed tonnage per year, set in advance, whatever the stock happens to be. Swap the fraction for a
fixed catch of $H$ thousand fish per year:

$$u'=0.5\,u\Big(1-\frac{u}{10}\Big)-H.$$

Read it as a sentence: *the fish grow logistically, and every year you take $H$ thousand of them no
matter how many are left.*

In the left panel, subtracting $H$ slides the whole sideways graph to the left by $H$. Before you
touch the slider, predict:

1. As the graph slides left, what happens to the places where it crosses the center line?
2. Is there a largest quota the fishery can survive? The unharvested fish grow fastest at
   $u=5$, at a rate of $0.5(5)(0.5)=1.25$ thousand per year. Does that number matter?
3. As the quota approaches that limit, does the settled population shrink smoothly toward zero,
   like it did in section 2?

Run the cell and drag $H$ to both ends. Then go slowly through the range from $1.1$ to $1.4$, one
step at a time.

In [ ]:
from ipywidgets import interact, FloatSlider

def explore_quota(H=0.8):
    H = round(H, 2)
    f = lambda u: 0.5*u*(1 - u/10) - H
    portrait(f, u_range=(-2, 12.5), starts=[1, 3, 6, 11], t_end=60, f_lim=2.5,
             physical_min=0, floor=0,
             title=f"u' = 0.5u(1 - u/10) - {H:.2f}   (quota H = {H:.2f} thousand fish per year)")

interact(
    explore_quota,
    H=FloatSlider(value=0.8, min=0.0, max=1.6, step=0.05,
                  description='H', continuous_update=False)
);

### What changed, and what stayed the same?

**In your notes,** fill these in from the slider:

- At $H=0.8$ there are two fixed points, at ________ (stable) and ________ (unstable).
- At $H=1.20$, a population that starts at $11$ settles at ________.
- At $H=1.30$, the same population ________.
- What does the lower fixed point mean for a fishery manager?

<details>
<summary>Check yourself after you have written something</summary>

At $H=0.8$ the fixed points are $8$ (stable) and $2$ (unstable). As $H$ grows they move toward each
other, meet at $u=5$ when $H=1.25$, and then vanish. At that exact moment the dot is half filled,
the same picture as section 3, and the same half-filled dot you saw at the threshold in section 2.

At $H=1.20$ a population starting at $11$ settles at $6$, a healthy-looking fishery. At $H=1.30$ the
same population goes extinct. Nudging the quota by $0.1$ took the fishery from thriving to gone,
and the settled level never passed through anything small on the way. Look at how it dies, too:
the curve that starts at $11$ sits between $4$ and $6$ for about thirty years, looking every bit like
a stable fishery, and then drops to zero around year $56$. Anyone watching that stock for a decade
would have called it sustainable.

The lower fixed point is a <strong>tipping point</strong>. Even under a legal quota, a bad year that
knocks the stock below it means collapse, because every arrow below it points down.
</details>

This is the answer to the question at the end of section 3. A semi-stable point is what two fixed
points look like at the instant they collide and disappear. That is why "semi-stable" counts as
unstable. It is the most fragile situation an equation can be in, and one nudge to the parameter
wipes it out.

The two harvesting rules take the same fish and fail in completely different ways, and you found
both of them from the graph of $f$ without solving either equation.

## 5. Final challenge: read this one cold

No scaffolding here. Work it on paper first, then run the cell.

Some species need company. Below a certain density, animals cannot find mates or defend against
predators, and the population declines even with plenty of room to grow. Biologists call this the
**Allee effect**, and one model for it is

$$u'=0.02\,u\,(u-2)(10-u).$$

Work out, on paper:

1. Where are the fixed points?
2. Which way do the arrows point on each interval for $u\ge 0$?
3. Classify each fixed point.
4. A population that starts at $u(0)=1.8$ and one that starts at $u(0)=2.2$. Where does each end
   up?
5. If you were writing a conservation plan for this species, which fixed point would you care about
   most, and why?

Commit to answers. Then run the cell.

In [ ]:
portrait(lambda u: 0.02*u*(u - 2)*(10 - u), u_range=(-1, 13),
         starts=[0.5, 1.8, 2.2, 5, 12], t_end=20, f_lim=2.5, physical_min=0,
         title="u' = 0.02u(u - 2)(10 - u)")

<details>
<summary>Check yourself once you have written your answers down</summary>

Fixed points at $u=0$, $u=2$, and $u=10$. Between $0$ and $2$, $f<0$, so populations fall. Between
$2$ and $10$, $f>0$, so they rise. Above $10$, $f<0$, so they fall back. That makes $0$ and $10$
stable and $2$ unstable: the same stable, unstable, stable pattern as the build-it-backwards problem
from the class packet.

Starting at $1.8$ the population dies out; starting at $2.2$ it recovers all the way to $10$. Two
populations only $400$ animals apart have opposite futures. The point that matters most for
conservation is the unstable one at $u=2$: keep the population above it and nature does the rest,
let it slip below and no amount of habitat will save it.
</details>

## Wrap-up

You predicted the long-run fate of five different equations, found a collapse threshold, and found
a tipping point, and you did not solve a single one of them.

The whole method is the one sentence from the top. Given an autonomous equation $u'=f(u)$, graph
$f$, find where it crosses zero, and read the sign in between. The phase line is what is left of
every solution once you throw away the timing and keep the destination.

Some things the phase line shows you are nearly invisible in a formula even when you have one. The
slowdown before the fraction fishery collapsed, and the quota fishery going from healthy to extinct
without ever looking small, are both there in the solution formulas if you dig. In the picture
they are the first thing you see. When you move on to equations with no formula at all, the
picture still works.

### Exit question

Sections 2 and 4 harvest the same fish. In two or three sentences, explain why taking a fixed
fraction lets the population fade gradually as you increase the harvest, while taking a fixed quota
can make it crash all at once. Use the phrase "fixed points" in your answer.